In [ ]:
#| default_exp core.autograd
#| export

import numpy as np
from typing import Optional, List, Tuple
import sys
import os

from tinytorch.core.tensor import Tensor

EPSILON = 1e-7

In [ ]:
#| export
class Function:
    """
    Base class for differentiable operations.

    Every operation that needs gradients (add, multiply, matmul, etc.)
    will inherit from this class and implement the apply() method.

    **Key Concepts:**
    - **saved_tensors**: Store inputs needed for backward pass
    - **apply()**: Compute gradients using chain rule

    **Example Usage:**
    ```python
    class AddBackward(Function):
        def apply(self, grad_output):
            # Addition distributes gradients equally
            return grad_output, grad_output
    ```
    """

    def __init__(self, *tensors):
        """
        Initialize function with input tensors.

        Args:
            *tensors: Input tensors that will be saved for backward pass
        """
        self.saved_tensors = tensors

    def apply(self, grad_output):
        """
        Compute gradients for inputs.

        Args:
            grad_output: Gradient flowing backward from the output

        Returns:
            Tuple of gradients for each input tensor

        **Must be implemented by subclasses**
        """
        raise NotImplementedError("Each Function must implement apply() method")

In [ ]:
#| export
class AddBackward(Function):
    """
    Gradient computation for tensor addition.

    **Mathematical Rule:** If z = a + b, then ∂z/∂a = 1 and ∂z/∂b = 1

    **Key Insight:** Addition distributes gradients equally to both inputs.
    The gradient flowing backward is passed unchanged to each input.

    **Broadcasting Handling:** When input shapes differ due to broadcasting,
    we sum gradients appropriately to match original tensor shapes.
    """

    def apply(self, grad_output):
        """
        Compute gradients for addition.

        Args:
            grad_output: Gradient flowing backward from output

        Returns:
            Tuple of (grad_a, grad_b) for the two inputs

        **Mathematical Foundation:**
        - ∂(a+b)/∂a = 1 → grad_a = grad_output
        - ∂(a+b)/∂b = 1 → grad_b = grad_output

        TODO: Implement gradient computation for addition operation.

        APPROACH:
        1. Extract input tensors from self.saved_tensors
        2. Initialize grad_a and grad_b to None
        3. For first input (a): if it requires gradients, set grad_a = grad_output
        4. For second input (b): if it requires gradients, set grad_b = grad_output
        5. Return tuple (grad_a, grad_b)

        EXAMPLE:
        >>> a = Tensor([1, 2, 3], requires_grad=True)
        >>> b = Tensor([4, 5, 6], requires_grad=True)
        >>> z = a + b  # z = [5, 7, 9]
        >>> # During backward: grad_output = [1, 1, 1]
        >>> # Result: grad_a = [1, 1, 1], grad_b = [1, 1, 1]

        HINTS:
        - Addition distributes gradients equally (derivative of a+b w.r.t. both is 1)
        - Check isinstance(tensor, Tensor) and tensor.requires_grad before computing
        - Return None for inputs that don't require gradients
        """

        a, b = self.saved_tensors
        grad_a = grad_b = None

        # gradient for first input
        if isinstance(a, Tensor) and a.requires_grad:
            grad_a = grad_output

        # gradient for second input
        if isinstance(b, Tensor) and b.requires_grad:
            grad_b = grad_output

        return grad_a, grad_b

In [ ]:
#| export
class MulBackward(Function):
    """
    Gradient computation for tensor multiplication.

    **Mathematical Rule:** If z = a * b, then ∂z/∂a = b and ∂z/∂b = a

    **Key Insight:** Each input's gradient equals the gradient output
    multiplied by the OTHER input's value (product rule).

    **Applications:** Used in weight scaling, binary masking,
    and anywhere element-wise multiplication occurs.
    """

    def apply(self, grad_output):
        """
        Compute gradients for multiplication.

        Args:
            grad_output: Gradient flowing backward from output

        Returns:
            Tuple of (grad_a, grad_b) for the two inputs

        **Mathematical Foundation:**
        - ∂(a*b)/∂a = b → grad_a = grad_output * b
        - ∂(a*b)/∂b = a → grad_b = grad_output * a

        TODO: Implement gradient computation for element-wise multiplication.

        APPROACH:
        1. Extract input tensors a, b from self.saved_tensors
        2. Initialize grad_a and grad_b to None
        3. For first input (a): if requires_grad, compute grad_a = grad_output * b
        4. For second input (b): if requires_grad, compute grad_b = grad_output * a
        5. Handle both Tensor and scalar cases for b
        6. Return tuple (grad_a, grad_b)

        EXAMPLE:
        >>> a = Tensor([2, 3], requires_grad=True)
        >>> b = Tensor([4, 5], requires_grad=True)
        >>> z = a * b  # z = [8, 15]
        >>> # During backward: grad_output = [1, 1]
        >>> # grad_a = [1, 1] * [4, 5] = [4, 5]
        >>> # grad_b = [1, 1] * [2, 3] = [2, 3]

        HINTS:
        - Product rule: each input's gradient equals grad_output times the OTHER input
        - Check if b is a Tensor or scalar before accessing .data
        - Use b.data if Tensor, or b directly if scalar
        """

        a, b = self.saved_tensors
        grad_a = grad_b = None

        # gradient for first input: grad_output * b
        if isinstance(a, Tensor) and a.requires_grad:
            if isinstance(b, Tensor):
                grad_a = grad_output * b.data
            else:
                grad_a = grad_output * b
        
        # gradient for second input: grad_output * a
        if isinstance(b, Tensor) and b.requires_grad:
            grad_b = grad_output * a.data

        return grad_a, grad_b

In [ ]:
#| export
class SubBackward(Function):
    """
    Gradient computation for tensor subtraction.

    **Mathematical Rule:** If z = a - b, then ∂z/∂a = 1 and ∂z/∂b = -1
    """

    def apply(self, grad_output):
        """
        Compute gradients for subtraction.

        Returns:
            Tuple of (grad_a, grad_b) where grad_b is negated

        TODO: Implement gradient computation for subtraction operation.

        APPROACH:
        1. Extract input tensors from self.saved_tensors
        2. Initialize grad_a and grad_b to None
        3. For first input (a): if requires_grad, set grad_a = grad_output
        4. For second input (b): if requires_grad, set grad_b = -grad_output (note the negative!)
        5. Return tuple (grad_a, grad_b)

        EXAMPLE:
        >>> a = Tensor([5, 7], requires_grad=True)
        >>> b = Tensor([2, 3], requires_grad=True)
        >>> z = a - b  # z = [3, 4]
        >>> # During backward: grad_output = [1, 1]
        >>> # grad_a = [1, 1], grad_b = -[1, 1] = [-1, -1]

        HINTS:
        - ∂(a-b)/∂a = 1 (gradient flows unchanged to first operand)
        - ∂(a-b)/∂b = -1 (gradient is negated for second operand)
        - The negative sign is crucial for correct gradient flow
        """

        a, b = self.saved_tensors
        grad_a, grad_b = None

        if isinstance(a, Tensor) and a.requires_grad:
            grad_a = grad_output

        if isinstance(b, Tensor) and b.requires_grad:
            grad_b = -grad_output

        return grad_a, grad_b

In [ ]:
#| export
class DivBackward(Function):
    """
    Gradient computation for tensor division.

    **Mathematical Rule:** If z = a / b, then:
    - ∂z/∂a = 1/b
    - ∂z/∂b = -a/b²
    """

    def apply(self, grad_output):
        """
        Compute gradients for division using quotient rule.

        Returns:
            Tuple of (grad_a, grad_b)

        TODO: Implement gradient computation for division operation.

        APPROACH:
        1. Extract input tensors from self.saved_tensors
        2. Initialize grad_a and grad_b to None
        3. For first input (a): if requires_grad, compute grad_a = grad_output / b
        4. For second input (b): if requires_grad, compute grad_b = -grad_output * a / (b²)
        5. Handle both Tensor and scalar cases for b
        6. Return tuple (grad_a, grad_b)

        EXAMPLE:
        >>> a = Tensor([8.0, 12.0], requires_grad=True)
        >>> b = Tensor([2.0, 3.0], requires_grad=True)
        >>> z = a / b  # z = [4.0, 4.0]
        >>> # During backward: grad_output = [1, 1]
        >>> # grad_a = [1, 1] / [2, 3] = [0.5, 0.333...]
        >>> # grad_b = -[1, 1] * [8, 12] / ([2, 3]²) = [-2, -1.333...]

        HINTS:
        - Quotient rule: ∂(a/b)/∂a = 1/b, ∂(a/b)/∂b = -a/b²
        - Use b.data if Tensor, or b directly if scalar
        - b² means b.data ** 2 for tensors
        """

        a, b = self.saved_tensors
        grad_a, grad_b = None

        if isinstance(a, Tensor) and a.requires_grad:
            if isinstance(b, Tensor):
                grad_a = grad_output / b.data
            else:
                grad_a = grad_output / b
                
        if isinstance(b, Tensor) and b.requires_grad:
            grad_b = -grad_output * a.data / (b.data ** 2)
        
        return grad_a, grad_b